# 04. Formula OCR (Model A - Step 3)

`02_layout_detection.ipynb`에서 감지한 `formula` 영역을 LaTeX로 변환합니다.

**입력:** `outputs/text_ocr/batch_text.json` (03에서 text 채운 결과)  
**출력:** `formula` 요소의 `latex` 필드 채운 JSON

In [ ]:
!pip install pix2tex -q

In [ ]:
import fitz
import io
import os
import json
from PIL import Image, ImageOps, ImageFilter
import matplotlib.pyplot as plt

PDF_PATH   = "../data/sample/2015개정경제수학-광주교육청.pdf"
TEXT_DIR   = "../outputs/text_ocr"
OUTPUT_DIR = "../outputs/formula_ocr"
os.makedirs(OUTPUT_DIR, exist_ok=True)

doc = fitz.open(PDF_PATH)
print(f"총 페이지 수: {len(doc)}")

In [ ]:
def page_to_image(doc, page_num, dpi=300):
    page = doc[page_num]
    pix = page.get_pixmap(dpi=dpi)
    return Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")

def crop_element(img, bbox_px, pad=8):
    x1, y1, x2, y2 = bbox_px
    x1, y1 = max(0, x1 - pad), max(0, y1 - pad)
    x2, y2 = min(img.width, x2 + pad), min(img.height, y2 + pad)
    return img.crop((x1, y1, x2, y2))

def preprocess_formula(crop):
    """수식 이미지: 흑백 + 여백 추가 (pix2tex 권장 전처리)"""
    gray = ImageOps.grayscale(crop)
    # 흰 배경에 검정 수식이어야 인식률이 높음
    if gray.getextrema()[0] > 128:  # 배경이 밝으면 반전 불필요
        pass
    padded = ImageOps.expand(gray, border=10, fill=255)
    return padded

## 1. 모델 로드

첫 실행 시 모델 다운로드 (~1.3GB).

In [ ]:
from pix2tex.cli import LatexOCR

model = LatexOCR()
print("모델 로드 완료")

## 2. 단일 페이지 테스트

In [ ]:
TEST_PAGE = 11

text_path = os.path.join(TEXT_DIR, f"page_{TEST_PAGE+1}_text.json")
with open(text_path, encoding="utf-8") as f:
    page_data = json.load(f)

formula_elements = [e for e in page_data["elements"] if e["type"] == "equation"]
print(f"수식 영역: {len(formula_elements)}개")

if not formula_elements:
    print("이 페이지에 수식 없음 — 다른 페이지로 TEST_PAGE 변경해보세요")

In [ ]:
img = page_to_image(doc, TEST_PAGE)

for elem in formula_elements:
    crop = crop_element(img, elem["bbox_px"])
    processed = preprocess_formula(crop)
    try:
        latex = model(processed)
    except Exception as e:
        latex = f"ERROR: {e}"
    elem["latex"] = latex
    print(f"[{elem['id']}] → {latex}")

## 3. 크롭 + LaTeX 시각화

In [ ]:
if formula_elements:
    n = len(formula_elements)
    fig, axes = plt.subplots(n, 1, figsize=(10, n * 2.5))
    if n == 1:
        axes = [axes]

    for i, elem in enumerate(formula_elements):
        crop = crop_element(img, elem["bbox_px"])
        axes[i].imshow(crop)
        axes[i].axis("off")
        axes[i].set_title(f"{elem['id']}\nLaTeX: {elem.get('latex', '')}", fontsize=9)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"page_{TEST_PAGE+1}_formulas.png"), dpi=120, bbox_inches="tight")
    plt.show()

## 4. JSON 업데이트 저장

In [ ]:
out_path = os.path.join(OUTPUT_DIR, f"page_{TEST_PAGE+1}_formula.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(page_data, f, ensure_ascii=False, indent=2)

print(f"저장 완료: {out_path}")

## 5. 배치 처리

In [ ]:
batch_text_path = os.path.join(TEXT_DIR, "batch_text.json")
with open(batch_text_path, encoding="utf-8") as f:
    all_pages = json.load(f)

for page_data in all_pages:
    page_num = page_data["page_id"] - 1
    formula_elems = [e for e in page_data["elements"] if e["type"] == "equation"]

    if not formula_elems:
        print(f"[{page_num+1}페이지] 수식 없음 — 스킵")
        continue

    print(f"[{page_num+1}페이지] 수식 {len(formula_elems)}개 처리 중...", end=" ")
    img = page_to_image(doc, page_num)

    for elem in formula_elems:
        crop = crop_element(img, elem["bbox_px"])
        processed = preprocess_formula(crop)
        try:
            elem["latex"] = model(processed)
        except Exception as e:
            elem["latex"] = f"ERROR: {e}"

    print("완료")

batch_out = os.path.join(OUTPUT_DIR, "batch_formula.json")
with open(batch_out, "w", encoding="utf-8") as f:
    json.dump(all_pages, f, ensure_ascii=False, indent=2)

print(f"\n배치 저장 완료: {batch_out}")